In [0]:
import re
import pandas as pd

def standardize_street_type(df, addr_col):

    df = df.copy()

    df[addr_col] = df[addr_col].str.strip().str.upper()       # 

    # first split the address column into line 1 and line 2 
    # based on the actual street address and the specific floor or suite details

    split_pattern = r'\b(SUITE|STE|APT|UNIT|BLDG|BUILDING|FLOOR|FL| # )\b'        # save the pattern for splitting the address column
    def split_address(addr):
        if pd.isna(addr):
            return '', ''
        
        """Split address into line 1 and line 2"""
        match = re.search(split_pattern, addr)

        if match:
            cutoff_idx = match.start()
            
            line_1 = addr[:cutoff_idx].strip().rstrip(",")
            line_2 = addr[cutoff_idx:].strip()
        else:
            line_1 = addr
            line_2 = ''

        return line_1, line_2

    df[[addr_col + '_line_1', addr_col + '_line_2']] = pd.DataFrame(df[addr_col].apply(split_address).tolist(), index=df.index)


    """Standardize common street type abbreviations"""
    street_type_mapping = {
        'ST': 'Street', 'STR': 'Street', 'STREET': 'Street', 'ST.,': 'Street',
        'AVE': 'Avenue', 'AV': 'Avenue', 'AVENUE': 'Avenue',
        'RD': 'Road', 'RD.': 'Road', 'ROAD': 'Road',
        'BLVD': 'Boulevard', 'BLVD.': 'Boulevard', 'BOULEVARD': 'Boulevard',
        'DR': 'Drive', 'DR.': 'Drive', 'DRIVE': 'Drive',
        'LN': 'Lane', 'LN.': 'Lane', 'LANE': 'Lane',
        'CT': 'Court', 'CT.': 'Court', 'COURT': 'Court',
        'PL': 'Place', 'PL.': 'Place', 'PLACE': 'Place', 'PLZ': 'Plaza',
        'WAY': 'Way', 'WAY': 'Way',
        'PKWY': 'Parkway', 'PKWY.': 'Parkway', 'PARKWAY': 'Parkway',
    }
    
    # set the function to standardize street types
    def standardize_street_type(addr):
        """Standardize common street type abbreviations"""
        words = addr.split()
        
        if words and words[-1] in street_type_mapping:
            words[-1] = street_type_mapping[words[-1]]
            addr = ' '.join(words)
        return addr.upper()
    
    # apply the function and save the updated values
    df[addr_col + '_line_1'] = df[addr_col + '_line_1'].apply(standardize_street_type)
    
# Fix the directional abbreviations
    def standardize_direction(addr):

        """Standardize directional indicators"""
        dir_mapping = {
            'N': 'NORTH',
            'S': 'SOUTH',
            'E': 'EAST', 
            'W': 'WEST', 
            'NE': 'NORTHEAST', 
            'NW': 'NORTHWEST', 
            'SE': 'SOUTHEAST', 
            'SW': 'SOUTHWEST', 
        }

        # directional indicators
        direct_indic = r'\b(N|NORTH|S|SOUTH|E|EAST|W|WEST|NE|NORTHEAST|NW|NORTHWEST|SE|SOUTHEAST|SW|SOUTHWEST)\b'

        match = re.search(direct_indic, addr)
        if match:
            direction = match.group()
            addr = re.sub(direct_indic, dir_mapping[direction], addr)
        
        return addr.upper()
    
    df[addr_col + '_line_1'] = df[addr_col + '_line_1'].apply(standardize_direction)


    def standardize_addr_2(addr):
        """Standardize common street type abbreviations"""

        addr_2_mapping = {
        'APT': 'APARTMENT', 'APT.': 'APARTMENT', 'APARTMENT': 'APARTMENT',
        'APT#': 'APARTMENT', 'APT.#': 'APARTMENT',
        'FL': 'FLOOR', 'FL.': 'FLOOR', 'FLOOR': 'FLOOR',
        'RM': 'ROOM', 'RM.': 'ROOM', 'ROOM': 'ROOM',
        'SUITE': 'SUITE', 'SUITE.': 'SUITE', 'STE': 'SUITE', 'STE.': 'SUITE',
        'UNIT': 'UNIT', 'UNIT.': 'UNIT',
        "#": 'SUITE'
            }
        
        pattern = "|".join(addr_2_mapping.keys())
        
        match = re.search(pattern, addr)
        if match:
            addr_2 = match.group()
            addr = re.sub(pattern, addr_2_mapping[addr_2], addr)
        
        return addr
    
    df[addr_col + '_line_2'] = df[addr_col + '_line_2'].apply(standardize_addr_2)
    
    return df

Import the bronze data

In [0]:
bronze_df = spark.read.table("cms_mdm_bronze.cms_mdm_ingest.ca_pediatrics_data").toPandas()
display(bronze_df)


In [0]:
bronze_updated_address_df = standardize_street_type(df=bronze_df, addr_col='Practice_Address')
display(bronze_updated_address_df)


standardize the phone numbers

In [0]:
import re

def clean_phone(num):
    if pd.isna(num):
        return '', '', False
    
    raw = str(num)

    # Extract extension
    ext_match = re.search(r'(ext\.?|x|#)\s*(\d+)$', raw, flags=re.I)
    ext = ext_match.group(2) if ext_match else ''

    # Remove extension from main number
    raw = re.sub(r'(ext\.?|x|#)\s*\d+$', '', raw, flags=re.I)

    # Remove non-digits
    digits = re.sub(r'\D', '', raw)

    # Normalize US numbers
    if len(digits) == 11 and digits.startswith('1'):
        digits = digits[1:]

    valid = len(digits) == 10

    return digits, ext, valid


In [0]:
phone_cols = pd.DataFrame(bronze_updated_address_df['Phone'].apply(clean_phone).tolist(), index=bronze_updated_address_df.index)
phone_cols.columns = ['Phone_std', 'Phone_Ext', 'Phone_Valid']
bronze_updated_addr_phone = pd.concat([bronze_updated_address_df, phone_cols], axis=1)

display(bronze_updated_addr_phone)


In [0]:
bronze_updated_addr_phone.columns

In [0]:
col_order = ['Legacy_System_ID', 'NPI', 'First_Name', 'Last_Name',
       'Practice_Address','Practice_Address_line_1',
       'Practice_Address_line_2', 'City', 'State', 'Zip', 'Phone', 'Phone_std', 'Phone_Ext', 'Phone_Valid',
       'Last_Updated_Date', 'Source_System' ]

In [0]:
bronze_updated_addr_phone = bronze_updated_addr_phone[col_order].drop(['Phone_Valid'],axis=1)

In [0]:
silver_df = spark.createDataFrame(bronze_updated_addr_phone)
display(silver_df)


In [0]:
silver_df.write.mode("overwrite").saveAsTable("cms_mdm_silver.cms_mdm_silver.ca_pediatrics_silver_data")